# PACE access & spectrum extraction (Stage 3)

PAB reads PACE/OCI L2 AOP granules **cloud-first** and extracts the ~10 nearest **unflagged** `Rrs` spectra around each float. This notebook demonstrates the offline pieces — `l2_flags` decoding and nearest-pixel extraction on a small synthetic granule — and ends with an **optional live** granule discovery via `earthaccess`.

See `docs/pace_access.rst` for the cloud-read (lazy-S3 vs OPeNDAP) trade-off.

In [1]:
import numpy as np
import xarray as xr

## 1. Decoding `l2_flags`

`pab.pace.flags` decodes the SeaDAS/OB.DAAC bitmask. PAB's default screen is the standard ocean mask from the design.

In [2]:
from pab.pace import flags

val = flags.flag_value(['LAND', 'CLDICE'])
print('combined value:', val)
print('decoded       :', flags.decode(val))
print('standard mask :', flags.STANDARD_OCEAN_MASK)

combined value: 514
decoded       : ('LAND', 'CLDICE')
standard mask : ('ATMFAIL', 'LAND', 'HIGLINT', 'HILT', 'STRAYLIGHT', 'CLDICE', 'COCCOLITH', 'HISATZEN', 'HISOLZEN', 'LOWLW', 'CHLFAIL', 'NAVFAIL', 'MAXAERITER')


## 2. A synthetic granule

The canonical granule dataset has dims `(x, y, wl)` for `Rrs`/`Rrs_unc`, 2-D `latitude`/`longitude`, a `wavelength` axis, and an `l2_flags` variable. Here `Rrs[i,j,:] = i*10 + j` so we can see which pixel is selected. We flag the centre pixel as `LAND`.

In [3]:
nx, ny = 5, 5
lat = np.linspace(44.0, 45.0, nx)
lon = np.linspace(-31.0, -30.0, ny)
lons2d, lats2d = np.meshgrid(lon, lat)
wave = np.array([440., 490., 550., 670.])
rrs = np.array([[[i*10 + j]*wave.size for j in range(ny)] for i in range(nx)],
               dtype=float)
l2 = np.zeros((nx, ny), dtype=np.int64)
l2[2, 2] = flags.flag_value(['LAND'])   # flag the centre pixel
ds = xr.Dataset(
    {'Rrs': (('x','y','wl'), rrs),
     'Rrs_unc': (('x','y','wl'), rrs*0.1),
     'l2_flags': (('x','y'), l2)},
    coords={'latitude': (('x','y'), lats2d),
            'longitude': (('x','y'), lons2d),
            'wavelength': ('wl', wave)})
ds

<xarray.Dataset> Size: 2kB
Dimensions:     (x: 5, y: 5, wl: 4)
Coordinates:
    latitude    (x, y) float64 200B 44.0 44.0 44.0 44.0 ... 45.0 45.0 45.0 45.0
    longitude   (x, y) float64 200B -31.0 -30.75 -30.5 ... -30.5 -30.25 -30.0
    wavelength  (wl) float64 32B 440.0 490.0 550.0 670.0
Dimensions without coordinates: x, y, wl
Data variables:
    Rrs         (x, y, wl) float64 800B 0.0 0.0 0.0 0.0 ... 44.0 44.0 44.0 44.0
    Rrs_unc     (x, y, wl) float64 800B 0.0 0.0 0.0 0.0 0.1 ... 4.4 4.4 4.4 4.4
    l2_flags    (x, y) int64 200B 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0

## 3. Nearest unflagged pixels

Target the centre of the granule. Because pixel (2,2) is flagged `LAND`, the extractor skips it and returns the nearest *valid* neighbours instead.

In [4]:
from pab.pace import extract

tlat = float(lats2d[2, 2]); tlon = float(lons2d[2, 2])
px = extract.nearest_valid_pixels(ds, tlat, tlon, n=5)
for p in px:
    print(f"rank {p['rank']}: pixel ({p['ix']},{p['iy']}) "
          f"dist={p['distance_km']:.2f} km")

rank 1: pixel (2,3) dist=19.83 km
rank 2: pixel (2,1) dist=19.83 km
rank 3: pixel (1,2) dist=27.80 km
rank 4: pixel (3,2) dist=27.80 km
rank 5: pixel (3,3) dist=34.12 km


In [5]:
# attach the spectra for the matchup engine (Stage 4)
recs = extract.extract_matchup_spectra(ds, tlat, tlon, n=3)
r0 = recs[0]
print('nearest valid pixel:', (r0['ix'], r0['iy']))
print('Rrs              :', r0['Rrs'])
print('wavelength       :', r0['wavelength'])

nearest valid pixel: (2, 3)
Rrs              : [23. 23. 23. 23.]
wavelength       : [440. 490. 550. 670.]


## 4. (Optional) live granule discovery

Set `RUN_LIVE = True` to query the Earthdata Cloud for real PACE granules around a point/time (requires an Earthdata Login; reading pixels is fast only in-region, AWS `us-west-2`). Skipped by default so the notebook runs offline.

In [6]:
RUN_LIVE = False  # flip to True (needs Earthdata Login / us-west-2)

if RUN_LIVE:
    import earthaccess
    from pab.pace import discover, cloud
    earthaccess.login(persist=True)
    results = discover.search_granules(
        temporal=('2024-05-01', '2024-05-08'),
        bounding_box=(-31, 44, -30, 45), cloud_cover=(0, 50))
    print(f'{len(results)} granules found')
    table = discover.granule_table(results)
    display(table.head())
    # read one granule lazily and extract near the box centre
    gds = cloud.open_granule(table.iloc[0]['data_url'], backend='s3')
    print(extract.nearest_valid_pixels(gds, 44.5, -30.5, n=3))
else:
    print('RUN_LIVE is False — skipping the network discovery.')

RUN_LIVE is False — skipping the network discovery.
